In [5]:
from huggingface_hub import InferenceClient
import os

# 1. Initialize the client with your free HF Token
# Get this at huggingface.co/settings/tokens
client = InferenceClient(token=os.environ.get("HF_TOKEN"))

# 2. Define the specific model you want to test (e.g., a sentiment classifier)
model_id = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# 3. Make the API Call
try:
    response = client.text_classification(
        "I'm feeling incredibly optimistic about fine-tuning my own models!",
        model=model_id
    )
    print(response)
    # Output: [{'label': 'positive', 'score': 0.98}, ...]
    
except Exception as e:
    print(f"Error: {e}")
    # This is where you would catch 503 Cold Start errors and retry

[TextClassificationOutputElement(label='positive', score=0.982831597328186), TextClassificationOutputElement(label='neutral', score=0.014756940305233002), TextClassificationOutputElement(label='negative', score=0.0024114681873470545)]


In [1]:
import os
from huggingface_hub import InferenceClient

# 1. Credential Injection
# It is best practice to set your token as an environment variable:
# Linux/Mac: export HF_TOKEN="your_hf_token_here"
# Windows: setx HF_TOKEN "your_hf_token_here"
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("WARNING: HF_TOKEN not found in environment. Please paste it below:")
    hf_token = input("Token: ").strip()

# Initialize the client
client = InferenceClient(token=hf_token)

# 2. Define the Target Architectures
# We are targeting the specific Instruct variants for Q&A formatting
PHI4_MODEL_ID = "microsoft/Phi-4-mini-instruct"
# We are targeting a smaller hosted model plus a larger one for comparison
QWEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LLAMA_MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
GPTOSS_MODEL_ID = "openai/gpt-oss-20b"
MIXRAL_MODEL_ID = "mistralai/Mixtral-8x22B-Instruct-v0.1"

# 3. Define the System and User Prompts
messages = [
    {"role": "system", "content": "You are a highly analytical AI assistant. Be concise."},
    {"role": "user", "content": "Explain the concept of 'Zero-Shot Learning' in one paragraph."}
]

def query_model(model_id, chat_messages):
    print(f"\n[{model_id}] Initializing connection...")
    try:
        # The chat_completion endpoint automatically formats the prompt 
        # to match the specific model's required chat template (e.g., <|user|>, [INST]).
        response = client.chat_completion(
            model=model_id,
            messages=chat_messages,
            max_tokens=150,
            temperature=0.1, # Low temperature for analytical consistency
        )
        
        # Extract the actual text from the response payload
        output = response.choices[0].message.content
        print(f"SUCCESS. Output:\n{output}\n")
        print("-" * 50)
        
    except Exception as e:
        print(f"FAILED. Error details: {e}\n")
        print("-" * 50)

# 4. Execute Operations
if __name__ == "__main__":
    print("Starting Model Inference Sequence...\n")
    
    # Test 1: Phi-4 Mini (4.0B Parameters)
    # This is a lightweight model and should run almost instantly on the free tier.
    query_model(PHI4_MODEL_ID, messages)
    
    # Test 2: Qwen 2.5 1.5B Instruct
    # This is a smaller hosted model with a working provider mapping.
    query_model(QWEN_MODEL_ID, messages)
    
    # Test 3: LLaMA 3 8B Instruct
    # This is a larger model that may trigger a cold start, but it is still within
    # the free tier limits and should be a good test of the system's handling of larger models.
    query_model(LLAMA_MODEL_ID, messages)
    
    # Test 4: GPT-OSS 20B
    query_model(GPTOSS_MODEL_ID, messages)
    
    # Test 5: Mixtral 8x22B (141B Total Parameters, 39B Active)
    # WARNING: See operational briefing below regarding this model.
    query_model(MIXRAL_MODEL_ID, messages)

Starting Model Inference Sequence...


[microsoft/Phi-4-mini-instruct] Initializing connection...
FAILED. Error details: 

--------------------------------------------------

[Qwen/Qwen2.5-1.5B-Instruct] Initializing connection...
SUCCESS. Output:
Zero-shot learning is an advanced machine learning technique where a model can perform tasks without being explicitly trained on examples from that task, relying instead on general knowledge and unsupervised features to make predictions or decisions. It's particularly useful for applications like image captioning, where it learns to describe images based on their content rather than specific labels provided during training. The key idea is to leverage pre-existing information about similar concepts across different domains to enable unseen tasks with minimal data.

--------------------------------------------------

[meta-llama/Meta-Llama-3-8B-Instruct] Initializing connection...
SUCCESS. Output:
Zero-Shot Learning (ZSL) is a machine learning